# Kanana-2-v-decide 3B (2611) — typed decisions on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/kanana-2-v-decide-typed-decisions-511b/colab/Kanana-2-v-decide-typed-decisions.ipynb)

Run Kakao's [`kakaocorp/kanana-2-v-decide-3b-2611`](https://huggingface.co/kakaocorp/kanana-2-v-decide-3b-2611)
with Transformers `AutoModel` + `model.system_one(...)`:
give it a **state** (text, JSON, or chat, optionally with images) and typed questions,
and get a **probability for every supplied option** — no text generation, zero output tokens.

This is a **research/demo notebook**. The model does not write replies. It returns
typed `choice` / `noul` / `score` answers over the options you pass in. Changing an
option changes the distribution.

**Why this model.** Kanana-2-v-decide is Kakao's **multimodal decision model**
(weights released `2026/11/06`). It is built on Kanana-2-v-3b: a **0.49B vision encoder**
plus a **3.18B language model** (**3.67B** total; Hub safetensors ~**7.33 GB** BF16).
Each question is encoded with the state into one embedding, each candidate into another;
a softmax over **cosine similarities** is the answer. The card reports **72.50** on
Kanana Decision Bench (Korean text) and **61.23** average on Decision Index + Decision Index Vision.

**Compared with** [`IOTA-Decide-4B-typed-decisions.ipynb`](IOTA-Decide-4B-typed-decisions.ipynb),
[`d1-2B-inference-decisions.ipynb`](d1-2B-inference-decisions.ipynb) /
[`d1-3B-inference-decisions.ipynb`](d1-3B-inference-decisions.ipynb) /
[`d1-Omni-2B-inference-decisions.ipynb`](d1-Omni-2B-inference-decisions.ipynb),
[`elephant-1-inference-decisions.ipynb`](elephant-1-inference-decisions.ipynb),
[`THX-01-decision-routing.ipynb`](THX-01-decision-routing.ipynb), and
[`pplx-decider-flash-v1.2-vs-d1.ipynb`](pplx-decider-flash-v1.2-vs-d1.ipynb):
same *job* (typed questions → probabilities), different checkpoint and head.
This notebook is Kakao's embedding/cosine `system_one` path, **Korean-strong**, with **image**
input. It does not load those other models.

**Licence:** [Kanana Open License](https://huggingface.co/kakaocorp/kanana-2-v-decide-3b-2611/blob/main/LICENSE)
(Hub `license: other`, `license_name: kanana-open-license`). Kakao's
[2026-11-06 press](https://www.kakaocorp.com/page/detail/12193) notes that commercial use
is allowed. Read the LICENSE file for the actual terms.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. A free T4 is Turing **sm_75**: ~15 GB
of VRAM, **no native bfloat16**, **no FlashAttention-2**. The card example uses
`dtype=torch.bfloat16`; this notebook loads **`dtype=torch.float16`** with **SDPA**
(eager if SDPA fails). Weights stay about **7.33 GB**; short demos should leave headroom
on 15 GB. Not measured on Colab in this environment (no GPU here).

**Not a chat model.** There is no `generate()`. Video is not supported. Keep a human in
the loop for high-stakes medical, legal, credit, or hiring decisions.

**Links:** [model card](https://huggingface.co/kakaocorp/kanana-2-v-decide-3b-2611) ·
[press](https://www.kakaocorp.com/page/detail/12193) ·
[blog (Korean)](https://tech.kakao.com/posts/846) ·
[demo Space](https://huggingface.co/spaces/hugging-apps/kanana-2-v-decide-3b) ·
[tweet source](https://x.com/aiupdatetoday/status/2118589305820258788)

## 1. Install

The [card `requirements.txt`](https://huggingface.co/kakaocorp/kanana-2-v-decide-3b-2611/blob/main/requirements.txt)
pins `torch==2.10.0`, `torchvision==0.25.0`, `transformers==5.5.3`, `pillow==12.1.1`,
`einops==0.8.2`, `safetensors==0.7.0` (checkpoint `transformers_version`: 5.5.3).

On Colab, install **transformers / pillow / einops / safetensors** at those pins, but
**do not reinstall torch or torchvision**. Colab already has a working CUDA wheel;
replacing it often breaks the T4 runtime. Use **`%pip`**, not `python -I -m pip`
(isolated mode drops Colab site-packages). Do **not** install `flash-attn`.

In [ ]:
%pip install -q "transformers==5.5.3" "pillow==12.1.1" "einops==0.8.2" "safetensors==0.7.0"

# Card also pins torch==2.10.0 and torchvision==0.25.0.
# Do not replace Colab's CUDA torch — that often breaks the T4 runtime.
# Use %pip, not `python -I -m pip` (isolated mode drops Colab site-packages).
# Do NOT install flash-attn (T4 / sm_75 cannot run FlashAttention-2).
print("install done — skipped torch/torchvision reinstall")

> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The pin to 5.5.3 is invisible until the process restarts.

## 2. Check the GPU

Print `torch.cuda.is_available()`, the GPU name, and free/allocated VRAM. T4 (`sm_75`) has
no native bfloat16 — **float16** is the path this notebook uses. The next cell stops if you
are on CPU.

In [ ]:
import gc
import json

import torch
import transformers
from packaging.version import Version

print("cuda available:", torch.cuda.is_available())
assert Version(transformers.__version__.split("+")[0]) >= Version("5.5.3"), (
    f"transformers {transformers.__version__} is loaded; kanana-2-v-decide needs 5.5.3 "
    "(card pin). Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Keep PyTorch SDPA on mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

DEVICE = "cuda"
major, minor = torch.cuda.get_device_capability()
try:
    NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    NATIVE_BF16 = major >= 8

# Card example uses torch.bfloat16. T4 has no native BF16 — stay on float16.
DTYPE = torch.float16
MODEL_ID = "kakaocorp/kanana-2-v-decide-3b-2611"

free_b, total_b = torch.cuda.mem_get_info()
allocated_b = torch.cuda.memory_allocated()
print("transformers", transformers.__version__, "| torch", torch.__version__)
print("device:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(
    f"VRAM free: {free_b / 1024**3:.2f} GB of {total_b / 1024**3:.1f} GB | "
    f"allocated: {allocated_b / 1024**3:.2f} GB"
)
print("dtype:", DTYPE, "| native bf16:", NATIVE_BF16)
print("FlashAttention-2: off (sdpa / eager only)")
if major < 8:
    print("T4 path: float16 + SDPA/eager. bfloat16 and FlashAttention-2 are off.")


def print_vram(prefix=""):
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"{prefix}VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB | "
        f"peak: {peak:.2f} GB | free: {free_now / 1024**3:.2f}/{total_now / 1024**3:.2f} GB"
    )

## 3. Load Kanana-2-v-decide

Official Transformers API from the [card](https://huggingface.co/kakaocorp/kanana-2-v-decide-3b-2611#transformers):
`AutoModel.from_pretrained(..., trust_remote_code=True, dtype=...)`. The first run downloads
about **7.33 GB**. The card uses bfloat16; this cell passes **float16** and prefers **SDPA**
(eager if SDPA refuses the stack).

A pipeline alternative exists (`pipeline("decision", ...)`) — this notebook stays on
`model.system_one` so the typed answers are easy to print.

In [ ]:
from transformers import AutoModel

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

load_kwargs = dict(trust_remote_code=True, dtype=DTYPE)
try:
    model = AutoModel.from_pretrained(
        MODEL_ID, attn_implementation="sdpa", **load_kwargs
    ).to(DEVICE)
except Exception as err:
    print("SDPA load failed, retrying eager:", type(err).__name__, err)
    model = AutoModel.from_pretrained(
        MODEL_ID, attn_implementation="eager", **load_kwargs
    ).to(DEVICE)

model.eval()

n_params = sum(p.numel() for p in model.parameters())
attn_impl = getattr(model.config, "_attn_implementation", None) or getattr(
    model.config, "attn_implementation", "default"
)
print("parameters:", f"{n_params / 1e9:.2f}B  (card: 3.67B = 0.49B vision + 3.18B LM)")
print("dtype:", next(model.parameters()).dtype)
print("attn implementation:", attn_impl)
print_vram("after load  ")


def show(response, title=None):
    if title:
        print(f"=== {title} ===")
    for name, answer in response["answers"].items():
        kind = answer.get("type")
        if kind == "noul":
            p_yes = answer["noul"]
            side = "yes" if p_yes >= 0.5 else "no"
            print(f"{name:>16}: noul   P(yes) = {p_yes:.3f}  →  {side}")
        elif kind == "choice":
            print(
                f"{name:>16}: choice  {answer['choice']}  "
                f"(confidence {answer['confidence']:.3f})"
            )
            for key, prob in answer["probabilities"].items():
                bar = "#" * int(round(prob * 20))
                print(f"{'':>16}    {key:22s} {prob:.3f}  {bar}")
        elif kind == "score":
            legend = answer.get("legend") or {}
            nearest = str(int(round(answer["score"])))
            label = legend.get(nearest, "")
            print(
                f"{name:>16}: score   {answer['score']:.3f}  "
                f"(confidence {answer['confidence']:.3f})  →  {label}"
            )
            for key, prob in answer["probabilities"].items():
                bar = "#" * int(round(prob * 20))
                print(f"{'':>16}    {key} {str(legend.get(key, '')):20s} {prob:.3f}  {bar}")
        else:
            print(f"{name:>16}: {answer}")
    print(f"{'usage':>16}: {response.get('usage')}")
    print()

## 4. Demo — support triage (card example)

The card's SystemOne example: a damaged-item replacement request. Three typed questions in
one `system_one` call. Each question is an embedding comparison, not generated text.

| `type` | What you get |
|---|---|
| `choice` | `choice` (option key), `probabilities`, `confidence` |
| `noul` | `noul` = P(yes) |
| `score` | `score` (expected level index), `probabilities`, `legend`, `confidence` |

`confidence` is the top probability minus the mean of the rest. Probabilities are **relative
to the options you supplied**.

In [ ]:
TRIAGE_STATE = "The item I ordered arrived damaged. Please replace it."
TRIAGE_QUESTIONS = {
    "category": {
        "type": "choice",
        "instructions": "Classify the customer's inquiry.",
        "criteria": {
            "delivery": "Delivery inquiry",
            "exchange": "Replacement request",
            "payment": "Payment inquiry",
        },
    },
    "damaged": {
        "type": "noul",
        "instructions": "Is the item damaged?",
    },
    "sentiment": {
        "type": "score",
        "instructions": "How positive is the customer's sentiment?",
        "criteria": ["Negative", "Neutral", "Positive"],
    },
}

triage = model.system_one(state=TRIAGE_STATE, questions=TRIAGE_QUESTIONS)
print(json.dumps(triage["answers"], indent=2))
print()
print("typed answers:")
print("  category:", triage["answers"]["category"]["choice"])
print("  damaged: ", triage["answers"]["damaged"]["noul"])
print("  sentiment:", triage["answers"]["sentiment"]["score"])
print()
show(triage, "support triage (text)")
print_vram("demo 1  ")

## 5. Demo — same noul question with a product photo

`system_one(..., images=[...])` accepts local paths, URLs, data URIs, or **PIL images**.
This cell draws a small cracked-phone still (no extra download) and asks the **same**
`"Is the item damaged?"` noul question against the photo.

In [ ]:
from PIL import Image, ImageDraw

phone = Image.new("RGB", (512, 384), (236, 230, 220))
draw = ImageDraw.Draw(phone)
draw.rounded_rectangle((140, 28, 372, 356), radius=28, fill=(28, 28, 32), outline=(12, 12, 12), width=4)
draw.rounded_rectangle((158, 56, 354, 300), radius=8, fill=(92, 148, 210))
draw.line([(176, 72), (250, 210), (330, 96)], fill=(248, 248, 252), width=5)
draw.line([(210, 88), (300, 268)], fill=(220, 228, 236), width=3)
draw.polygon([(240, 150), (318, 268), (228, 278)], fill=(18, 18, 22))
draw.ellipse((236, 312, 276, 340), fill=(60, 60, 66))
draw.rectangle((188, 168, 328, 214), fill=(176, 32, 36))
draw.text((214, 180), "DAMAGED", fill="white")

try:
    from IPython.display import display

    display(phone)
except Exception:
    phone.save("/tmp/kanana-damaged-phone.png")
    print("saved /tmp/kanana-damaged-phone.png")

photo = model.system_one(
    state="The customer attached a photo of the delivered item.",
    images=[phone],
    questions={"damaged": {"type": "noul", "instructions": "Is the item damaged?"}},
)
print(json.dumps(photo["answers"], indent=2))
print()
print("typed answer  damaged noul:", photo["answers"]["damaged"]["noul"])
print()
show(photo, "damaged? (product photo)")
print_vram("demo 2  ")

## 6. Demo — JSON-state routing

A dict/list state is treated as JSON (`state_format="json"` by default). Still one
`system_one` call: route the ticket, decide whether a human should look, score urgency.
The message is Korean — the card's Kanana Decision Bench is a Korean evaluation set.

In [ ]:
ROUTE_STATE = {
    "channel": "chat",
    "locale": "ko-KR",
    "order": {"id": "A-1842", "item": "무선 이어폰", "status": "delivered"},
    "message": "배송은 왔는데 결제가 두 번 된 것 같아요. 확인 부탁드려요.",
}
ROUTE_QUESTIONS = {
    "queue": {
        "type": "choice",
        "instructions": "Which queue should own this ticket?",
        "criteria": {
            "payments": "Duplicate charge, invoice, or refund",
            "delivery": "Tracking, missing, or late parcel",
            "product": "Item quality or how-to",
            "account": "Login or profile",
        },
    },
    "needs_human": {
        "type": "noul",
        "instructions": "Should a human agent review this before an automatic reply?",
    },
    "priority": {
        "type": "score",
        "instructions": "How soon should this be handled?",
        "criteria": ["Can wait", "Same day", "Immediate"],
    },
}

routed = model.system_one(state=ROUTE_STATE, questions=ROUTE_QUESTIONS)
print(json.dumps(routed["answers"], indent=2))
print()
print("typed answers:")
print("  queue:      ", routed["answers"]["queue"]["choice"])
print("  needs_human:", routed["answers"]["needs_human"]["noul"])
print("  priority:   ", routed["answers"]["priority"]["score"])
print()
show(routed, "JSON ticket routing")
print_vram("demo 3  ")

## 7. Caveats and cleanup

**What this notebook is for.** A free-T4 inference demo of Kanana-2-v-decide 3B (2611):
float16 + SDPA/eager, the official `AutoModel` + `system_one` API, a card-style text
triage, the same noul question on a tiny PIL product photo, and a JSON routing example.

**What it is not.** A chat model. A training notebook. A GGUF / llama.cpp / vLLM path.
Not a substitute for IOTA Decide, d1, elephant-1, THX-01, or pplx-decider — different
head (embedding/cosine) and a Korean-strong vision checkpoint.

**T4 / kernels.** float16 + SDPA (eager if needed). No native bfloat16, no FlashAttention-2.
Do not reinstall Colab's CUDA torch to match the card's `torch==2.10.0` pin.

**VRAM.** ~7.33 GB of float16 weights. These short states should fit a 15 GB T4. If a long
JSON state plus images OOMs, shorten the state (the processor already truncates over-long
inputs while keeping the beginning and end; images are kept). Max 10 images; no video.

**Calibration.** Answers are probabilities over *the options you passed*. Adding or renaming
an option changes the distribution. `temperature` (default 1.0) divides the logits.

**Licence.** [Kanana Open License](https://huggingface.co/kakaocorp/kanana-2-v-decide-3b-2611/blob/main/LICENSE).
See the LICENSE file rather than this notebook for terms.

**Links again:** [model card](https://huggingface.co/kakaocorp/kanana-2-v-decide-3b-2611) ·
[press](https://www.kakaocorp.com/page/detail/12193) ·
[tweet source](https://x.com/aiupdatetoday/status/2118589305820258788)

Free the GPU before you open another notebook:

In [ ]:
print_vram("before cleanup  ")
try:
    del model
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print_vram("after cleanup  ")
    print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print("Kanana-2-v-decide released. Re-run the load cell if you want another request.")